# The Amazing Race Dataset: Exploration & Statistical Analysis 🏁

Welcome to the starter exploration notebook for **The Amazing Race (TAR) Dataset**.

This notebook demonstrates how to:
1. Load tidy datasets stored in **Apache Parquet** format.
2. Calculate **Racing Averages (RA)** and identify the most dominant teams in race history.
3. Analyze **route itineraries, continents, and countries visited** across 36 seasons.
4. Track **leg-by-leg placements** and evaluate the impact of game mechanics (**Fast Forwards**, **U-Turns**, **Yields**).
5. Query the dataset using **zero-dependency SQLite** (`data/processed/tar.db`).
6. Interface with **Apache Arrow IPC** and **HuggingFace Datasets** for AI/LLM fine-tuning.

## 1. Setup & Environment

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd
import pyarrow.feather as feather

DATA_DIR = Path("../data/processed")
if not DATA_DIR.exists():
    DATA_DIR = Path("data/processed")

print(f"Using data directory: {DATA_DIR.resolve()}")

## 2. Load Processed Parquet Datasets

In [ ]:
seasons = pd.read_parquet(DATA_DIR / "seasons.parquet")
episodes = pd.read_parquet(DATA_DIR / "episodes.parquet")
teams = pd.read_parquet(DATA_DIR / "teams.parquet")
contestants = pd.read_parquet(DATA_DIR / "contestants.parquet")
legs = pd.read_parquet(DATA_DIR / "legs.parquet")
leg_results = pd.read_parquet(DATA_DIR / "leg_results.parquet")
tasks = pd.read_parquet(DATA_DIR / "tasks.parquet")

print(f"Loaded datasets:")
print(f"  • Seasons:     {len(seasons):,} records")
print(f"  • Episodes:    {len(episodes):,} records")
print(f"  • Teams:       {len(teams):,} records")
print(f"  • Contestants: {len(contestants):,} records")
print(f"  • Legs:        {len(legs):,} records")
print(f"  • Leg Results: {len(leg_results):,} records")
print(f"  • Tasks:       {len(tasks):,} records")

## 3. High-Level Season Overview

In [ ]:
# Summary table of the first 10 seasons
seasons[["season", "winners", "n_teams", "n_legs", "distance_miles"]].head(10)

In [ ]:
# Total distance raced across all seasons
total_miles = seasons["distance_miles"].sum()
avg_miles = seasons["distance_miles"].mean()
print(f"Total route distance traversed: {total_miles:,.0f} miles")
print(f"Average season route distance:  {avg_miles:,.0f} miles")

## 4. Racing Average (RA) Analysis

A team's **Racing Average** is the arithmetic mean of their finishing placements across all legs in a season.
Lower is better (1.0 is a theoretical clean sweep of every leg).

In [ ]:
# Calculate Racing Average for all teams with at least 10 completed legs
team_ra = (
    leg_results.groupby(["version", "season", "team_name"])["placement"]
    .agg(legs_raced="count", racing_average="mean")
    .reset_index()
)

# Merge with teams metadata to get final result and status
team_ra = team_ra.merge(teams[["season", "team_name", "result", "relationship", "status"]], on=["season", "team_name"], how="left")

# Top 15 Best Racing Averages in Amazing Race History
top_teams = team_ra[team_ra["legs_raced"] >= 10].sort_values("racing_average").head(15)
top_teams[["season", "team_name", "relationship", "legs_raced", "racing_average", "result", "status"]]

In [ ]:
# Compare average racing average of Winners vs. Runners-Up
winners_ra = team_ra[team_ra["result"] == 1]["racing_average"].mean()
runners_up_ra = team_ra[team_ra["result"] == 2]["racing_average"].mean()
third_place_ra = team_ra[team_ra["result"] == 3]["racing_average"].mean()

print(f"Mean Racing Average by Final Placement:")
print(f"  • 1st Place (Winners):    {winners_ra:.2f}")
print(f"  • 2nd Place (Runners-up): {runners_up_ra:.2f}")
print(f"  • 3rd Place (Finalists):  {third_place_ra:.2f}")

## 5. Route Itinerary & Geographic Analysis

In [ ]:
# Extract visited countries mentioned in route headers
import re

country_list = []
for route in legs["route_header"].dropna():
    parts = [p.strip() for p in re.split(r"→|->", route)]
    country_list.extend(parts)

country_counts = pd.Series(country_list).value_counts()
print("Top 15 Most Visited Destinations:")
print(country_counts.head(15))

## 6. Leg-by-Leg Progression & Hazard Impact

Let's trace how placements fluctuated across legs for the finalists of Season 1.

In [ ]:
s1_results = leg_results[leg_results["season"] == 1]
pivot_s1 = s1_results.pivot(index="leg_number", columns="team_name", values="placement")

# Display finalists
finalists = ["Rob & Brennan", "Frank & Margarita", "Joe & Bill"]
pivot_s1[[col for col in finalists if col in pivot_s1.columns]]

In [ ]:
# Total Fast Forwards, U-Turns, and Speed Bumps across the dataset
print(f"Total Fast Forwards claimed: {leg_results['fast_forward'].sum():,}")
print(f"Total U-Turn encounters:     {leg_results['uturn'].sum():,}")
print(f"Total Yield encounters:      {leg_results['yield'].sum():,}")
print(f"Total Speed Bumps completed: {leg_results['speed_bump'].sum():,}")

## 7. Zero-Dependency SQL Querying via SQLite (`tar.db`)

In [ ]:
db_path = DATA_DIR / "tar.db"
conn = sqlite3.connect(db_path)

query = """
SELECT 
    t.season,
    t.team_name,
    t.relationship,
    t.result,
    COUNT(r.placement) AS legs_raced,
    ROUND(AVG(r.placement), 2) AS racing_avg
FROM teams t
JOIN leg_results r ON t.version = r.version AND t.season = r.season AND t.team_name = r.team_name
WHERE t.result = 1
GROUP BY t.season, t.team_name
ORDER BY racing_avg ASC
LIMIT 10;
"""

top_winners_sql = pd.read_sql_query(query, conn)
conn.close()
top_winners_sql

## 8. Apache Arrow IPC & HuggingFace Datasets Integration

In [ ]:
# Direct zero-copy Arrow IPC loading
arrow_teams = feather.read_table(DATA_DIR / "arrow/teams.arrow")
print(f"Loaded Arrow Table with schema:\n{arrow_teams.schema}")
print(f"Total Arrow records: {arrow_teams.num_rows:,}")